# 1D J1J2J3 Inference: GRU variants (Euclidean, Poincare, Lorentz)

This notebook is part of the work arXiv:2604.24337 [quant-ph] (https://arxiv.org/html/2604.24337v1). In this notebook, I performed the inference process for the trained neural quantum states using 10000 samples. The loading directory in the Github repo might differ from the loading path used here. Please check and use the correct loading path. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('utility_lorentz')
from j1j2j3_train_loop_lorentz import *

sys.path.append('utility_poincare')
from j1j2j3_hyprnn_train_loop import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False
GPU Available:  False


In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    if 'Lorentz' in wf.name:
        test_samples_before = wf.sample_no_tau(numsamples)
    else: 
        test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_before, Marshall_sign=True)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        if 'Lorentz' in wf.name:
            test_samples_after = wf.sample_no_tau(numsamples)
        else: 
            test_samples_after = wf.sample(numsamples)        
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. Apply clipping
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3,Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
syssize = 100
numsamples = 10000
units =70
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fcn = 'trained_nqs/J1J2J3'

## J2=0.0, J3=0.5

In [4]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05=-53.9914

In [9]:
# EuclGRU: -38.2816
# with clipped e (1906 outliers - too many): -38.138099670410156+0.05909999832510948j), var E = 32.90729904174805
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (35.707298278808594-0.013899999670684338j), var E = 2.4951999187469482
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.28160095214844+0.05909999832510948j), var E = 61.37120056152344
DMRG energy  is -53.9914
Time taken=0.261 hrs


In [6]:
# PoincareGRU: 53.2067
# with clipped e (97 outliers): Mean E = (-53.190799713134766-0.007699999958276749j), var E = 3.0969998836517334
rmax=0.7
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypGRU_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.52450180053711-0.010300000198185444j), var E = 1.1734999418258667
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.20669937133789-0.007699999958276749j), var E = 3.335900068283081
DMRG energy  is -53.9914
Time taken=0.497 hrs


In [11]:
#LorentzGRU: single clamp: -53.8041 (with clipped E) --> Update using this new -53.8272
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.733001708984375-0.00139999995008111j), var E = 0.64410001039505
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.82720184326172+0.004999999888241291j), var E = 0.9156000018119812
DMRG energy  is -53.9914
Time taken=1.965 hrs


## J2=0.2, J3=0.2

In [24]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.5860

In [13]:
# EuclGRU: -40.8937
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (33.605499267578125-0.013500000350177288j), var E = 1.5256999731063843
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.893699645996094+0.012500000186264515j), var E = 1.3532999753952026
DMRG energy  is -43.586
Time taken=0.174 hrs


In [25]:
# with single clamp
#PoincareGRU: 
#with clipped E (677 outliers): Mean E = (-42.08679962158203-0.006300000008195639j), var E = 1.694100022315979
rmax=0.82
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypGRU_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (34.08890151977539-0.012600000016391277j), var E = 1.1059999465942383
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.03160095214844-0.006300000008195639j), var E = 2.8039000034332275
DMRG energy  is -43.586
Time taken=0.645 hrs


In [16]:
#LorentzGRU: single clamp:
#-42.7089 (with clipped E)
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (34.40299987792969-0.0007999999797903001j), var E = 0.4361000061035156
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.6797981262207+0.005200000014156103j), var E = 1.0760999917984009
DMRG energy  is -43.586
Time taken=2.474 hrs


## J2=0.2, J3=0.5

In [10]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [18]:
# EuclGRU: -48.8090
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (40.418399810791016-0.016100000590085983j), var E = 3.15939998626709
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.808998107910156+0.013100000098347664j), var E = 2.5146000385284424
DMRG energy  is -49.6287
Time taken=0.239 hrs


In [11]:
#PoincareGRU: -48.9628
#with clipped e (110 outliers): Mean E = (-48.95159912109375-0.0020000000949949026j), var E = 1.851099967956543
rmax=0.95 
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypGRU_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (40.99020004272461-0.025499999523162842j), var E = 2.7788000106811523
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.962799072265625-0.0020000000949949026j), var E = 1.9764000177383423
DMRG energy  is -49.6287
Time taken=0.64 hrs


In [19]:
#LorentzGRU: single clamp
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.608699798583984-0.0010000000474974513j), var E = 0.8223999738693237
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.124900817871094-0.00019999999494757503j), var E = 1.124899983406067
DMRG energy  is -49.6287
Time taken=3.081 hrs


## J2=0.5, J3=0.2

In [7]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [8]:
# EuclGRU
#without clipped e: Mean E = (-37.77560043334961+0.0003000000142492354j), var E = 0.48330000042915344
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/GRU_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (40.67219924926758-0.01679999940097332j), var E = 2.4870998859405518
Successfully remapped and loaded weights.
Clipped 428 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.77119827270508+0.0003000000142492354j), var E = 0.17820000648498535
DMRG energy  is -38.5473
Time taken=0.113 hrs


In [22]:
rmax=0.7 #single clamp default (forward pass)
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/GRU_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypGRU_70_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 15614
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.48320007324219-0.011800000444054604j), var E = 1.3085999488830566
Successfully remapped and loaded weights.
Clipped 496 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.31460189819336-0.00019999999494757503j), var E = 0.3370000123977661
DMRG energy  is -38.5473
Time taken=1.006 hrs


In [9]:
#LorentzGRU: single clamp
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/GRU_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.71649932861328-0.00019999999494757503j), var E = 0.6858999729156494
Successfully remapped and loaded weights.
Clipped 698 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.78030014038086+0.004999999888241291j), var E = 0.14319999516010284
DMRG energy  is -38.5473
Time taken=1.514 hrs
